[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/closed_universe/rendering.ipynb)

# Inside a Closed Universe

This example renders a closed universe, the round three-sphere, from inside: the 600-cell, whose 600 regular tetrahedra fill the sphere, flown through along a great circle, one of its Boerdijk–Coxeter helices panned past, and an irregular mesh of tetrahedra flown through. Light in the three-sphere runs along great circles, so each edge of a mesh, an arc of a great circle, is seen as a straight strut, and struts fill the view in every direction; past a quarter turn the great circles from the eye close in again on its antipode, and far struts loom larger. The 600-cell's 720 edges run along 72 great circles, ten edges to each, and a Boerdijk–Coxeter helix, a column of thirty tetrahedra stacked face to face, winds three of those circles about a fourth, its axis, and closes on itself once round the sphere.

Points of the three-sphere are antivectors of unit size in the algebra of four-dimensional space, the origin `mv.w.dual()` the pole of its equator `mv.w`. Every point carries an isoclinic frame, the motor `isoclinic(place)` that carries the origin to it by turning alike in two square planes. The three-sphere has such frames smoothly everywhere at once, which the two-sphere does not, and in them a ray's heading keeps still: from a place, a ray rolls along its great circle by `isoclinic(place) * ((equator ^ polar(heading)) * (-length / 2)).exp()`, where `equator ^ polar(heading)` is the rate that moves the origin along the heading. A camera is a motor too, carrying the origin to the eye and the origin's steps along `x`, `y` and `z` to the eye's right, up and forward: a turn after it flies it ahead along its own view, and a turn before it carries it with the whole view along a great circle. The renderer walks each ray through the mesh from tetrahedron to tetrahedron: a tetrahedron's ramps read a point's barycentric weights, and a ray leaves by the face across from the corner whose weight runs out first.

In the notation of Lie groups, the unit three-sphere is the group $\mathrm{SU}(2)$ with its bi-invariant metric, the isoclinic frames are its frame field of left translations $e_i(q) = q\,\tau_i$, and the geodesics from $q$ are the curves $q\exp(t\,\xi)$ for a fixed $\xi \in \mathfrak{su}(2)$, whose velocity $q \exp(t\,\xi)\,\xi$ has the same components $\xi$ in that frame all along.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from examples import instantiate
from examples.util.animation import show_animation
from examples.relativity.closed_universe import render, scenarios
from examples.relativity.closed_universe.core import ga

np.set_printoptions(precision=4, suppress=True)

mv = instantiate("examples.util.mesh", ga).mv
# Planes, and those through the origin, the polar planes of its steps; the rates that move and turn frames.
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within("x y z")                                          # Vector+
Rate = ga.gatype.bivector()                                                     # Bivector+
# Points of the sphere and the origin's steps; the motors that carry them.
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Point.derive.containing("w")                                             # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Translating = ga.gatype((Step, Through))                                        # Antivector- <- Vector+

# The origin's equator, the plane `w`, and the origin, its pole; the extrinsic pseudoscalar.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
pseudoscalar = mv.xyzw                                                          # [] Antiscalar+
# The step the origin takes under the motion across a plane through it, a step by its value, cast so; its inverse
# gives each step its polar plane.
polar = origin.commutator(equator ^ Through).cast(Translating).inverse()        # [] Vector+ <- Antivector-


def isoclinic(places: Point) -> Motor:
    """The isoclinic frames at places on the sphere, `Motor+`: the motors that carry the origin to each by turning
    alike in the plane of the great circle between them and in the plane square to it."""
    # The half of the quotient of the place by the origin that the pseudoscalar keeps, with one in the other half.
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## The 600-cell, flown through

The 600-cell's vertices are the 120 unit icosians, placed by a turn, and round, without waves, each of its tetrahedra is a piece of the sphere as it is. One of its Boerdijk–Coxeter helices gives the flight its path: the helix's axis threads a tunnel through its thirty tetrahedra, clear of every edge. A camera on the axis looks along it, and the turn that moves the origin along `z`, after the camera, flies it ahead; once round the sphere it is back where it started. Each great circle of the 600-cell's edges is drawn in its own colour. Drawing the flight takes about eight seconds, most of it tracing the frames.

In [ ]:
# The frames once round the sphere, and a frame's duration in milliseconds.
n_flight_frames = 90
duration_ms = 40

In [ ]:
# The 600-cell, and the waves the renderer reads, none; one of its Boerdijk–Coxeter helices: its rails, a point on its
# axis, and there the axis's direction and a step across it, steps of the origin in the frame at the point.
cell_mesh, cell_vertices = scenarios.six_hundred_cell()
still = scenarios.round_universe(cell_mesh, cell_vertices)
rails, on_axis, along, across_axis = scenarios.helix(cell_vertices)


def facing(placing: Motor, forward: Point, up: Point) -> Motor:
    """A camera: the motor `placing`, which carries the origin to the eye, after the turns that carry the origin's step
    along z to `forward` and then, about it, the one along y to `up`, unit steps of the origin square to each other in
    the eye's frame."""
    # The turn that carries one unit step to another in their plane is the square root of their quotient.
    aiming = (forward / mv.z.dual()).square_root()                              # [] Motor+
    rolling = (up / (aiming >> mv.y.dual())).square_root()                      # [] Motor+
    return placing * rolling * aiming


# The camera on the axis, looking along it with the step across it up, flown ahead along its view once round.
flight_start = facing(isoclinic(on_axis), along, across_axis)                   # [] Motor+
ahead = equator ^ polar(mv.z.dual().cast(Step))                                 # [] Rate+
distances = np.linspace(0, 2 * np.pi, n_flight_frames, endpoint=False)          # [frames]
flight = flight_start * (ahead * (-distances / 2)).exp()                        # [frames] Motor+
great_circles = render.edge_styles(cell_mesh, *render.great_polygon_struts(cell_mesh, cell_vertices))
show_animation(render.inside(cell_mesh, cell_vertices, still, flight, np.zeros(n_flight_frames), great_circles,
                             render.round_step, render.no_crystals(cell_mesh)), "closed_universe_flight", duration_ms)

## The helix, panned past

Seen from outside, the helix's thirty tetrahedra wind its three rails about the axis, linked by rungs. The camera stands most of a quarter turn out from the axis and looks back at it, the axis up and the view tilted. The turn in the axis's own plane carries the camera along the axis while it keeps every point of the square plane still, so the camera passes the helix without turning about it, and the rails corkscrew about the axis. The rails are drawn each in its own colour, the rungs light, and every other edge a thin ghost. Drawing the pan takes about four seconds, most of it tracing the frames.

In [ ]:
# The frames once round the axis; how far out from the axis the camera stands, and how far its view is tilted.
n_pan_frames = 60
pan_distance = 1.02
pan_tilt = 0.36

In [ ]:
# The camera that far out across the axis, looking back at it with the axis up, tilted about the view.
out_across = equator ^ polar(across_axis)                                        # [] Rate+
standing = isoclinic(on_axis) * (out_across * (-pan_distance / 2)).exp()         # [] Motor+
tilted = (out_across * pseudoscalar * (-pan_tilt / 2)).exp() >> along           # [] Point-
pan_start = facing(standing, -across_axis, tilted)                               # [] Motor+
# The turn along the axis: the rate that moves the point on it along its direction, carried there.
along_axis = isoclinic(on_axis) >> (equator ^ polar(along))                     # [] Rate+
angles = np.linspace(0, 2 * np.pi, n_pan_frames, endpoint=False)                # [frames]
pan = (along_axis * (-angles / 2)).exp() * pan_start                            # [frames] Motor+
show_animation(render.inside(cell_mesh, cell_vertices, still, pan, np.zeros(n_pan_frames),
                             render.edge_styles(cell_mesh, *render.helix_struts(cell_mesh, rails)),
                             render.round_step, render.no_crystals(cell_mesh)), "closed_universe_helix", duration_ms)

## An irregular mesh, flown through

The same flight runs through an irregular mesh of the sphere: the 600-cell's vertices and a few orbits of random points under the turns that carry the 600-cell onto itself, relaxed until its tetrahedra are as round as the orbits allow. Its edges follow no shared great circles and are drawn plain. A straight path through so many edges passes through some of them, and the renderer leaves out a strut while the eye is inside it. This cell takes about fifteen seconds, most of it relaxing the mesh and tracing the frames.

In [ ]:
irregular_mesh, irregular_vertices = scenarios.irregular_three_sphere()
# From the middle of its first tetrahedron, looking along the origin's step along z there, flown ahead once round.
middle = (irregular_mesh.incidence[3, 0].mean(axis=0) * irregular_vertices).batch()[0].normalized()   # [] Point-
irregular_flight = isoclinic(middle) * (ahead * (-distances / 2)).exp()         # [frames] Motor+
no_struts = np.zeros((0, 2), dtype=int), np.zeros((0, 3))
show_animation(render.inside(irregular_mesh, irregular_vertices,
                             scenarios.round_universe(irregular_mesh, irregular_vertices), irregular_flight,
                             np.zeros(n_flight_frames), render.edge_styles(irregular_mesh, *no_struts),
                             render.round_step, render.no_crystals(irregular_mesh)), "closed_universe_irregular", duration_ms)

In [ ]:
# checks
# The isoclinic frames carry the origin to each place; flown once round, the camera is back where it started; and the
# helix is thirty tetrahedra, every one with all four corners on its rails.
misses = (isoclinic(cell_vertices) >> origin) - cell_vertices                   # Point-[vertices]
np.testing.assert_allclose(misses.dot(misses).square_root().to_array(), 0, atol=1e-13)
once_round = (flight_start * (ahead * -np.pi).exp()) >> origin                  # [] Point-
back = once_round - (flight_start >> origin)                                    # [] Point-
np.testing.assert_allclose(back.dot(back).square_root().to_array(), 0, atol=1e-13)
assert np.isin(cell_mesh.tets, rails).all(axis=-1).sum() == 30